#Stock Analyzer

## This scrapes a news article about a Nigerian company from nairametrics.com then passes the news into an LLM for analysis which returns advice to a newbie investor on whether he should buy, sell or hold the company's shares.

In [ ]:
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from bs4 import BeautifulSoup
import requests

In [ ]:
# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    # print(soup.prettify())
    content = soup.find('div', class_='content-inner jeg_link_underline')
    return content.get_text() if content else "Content not found"

In [ ]:
system_prompt = """
You are seasoned stock analyst and speculator.
You are very good at analyzing news articles of a Nigerian company published on Nairametrics,
with the information in the article, you are able to give solid advice to a completely novice investor on whether to buy, sell or hold the stock of the company in question.

You always give your advice in this format:
# Company Name
## Summary of the news article in 2-3 sentences
## Advice to potential investors (Buy, Sell or Hold)
## Reasoning for the advice.

You always tailor your advice to suit novice investors only, giving a detailed explanation of the reasoning behind your advice.
Avoid using a lot of technical jargon, and if you must use technical terms, always explain them in simple terms.
"""

In [ ]:
user_prompt_prefix = """
Here are the contents of a company news article.
Provide an analysis of the news article.

"""

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openai = OpenAI()
# openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
def analyze_article(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        # model = "llama3.2:1b",
        # model = "gemma3:270m",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
def display_analysis(url):
    analysis = analyze_article(url)
    display(Markdown(analysis))

In [ ]:
display_analysis("https://nairametrics.com/2026/04/23/unilever-nigeria-plc-sustains-double-digit-growth-momentum-in-q1-2026-unaudited-results/")